In [1]:
%cd ~/IAA
%load_ext autoreload
%autoreload 2

/Users/woojinji/IAA


In [2]:
# Day 1 follow-up (Part 2): portfolio shift, renewal_count, missing values, policy steps. No model.
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.home() / "IAA"
sys.path.insert(0, str(ROOT))
from src.target import LINES, load_claims, load_policies, match_claims

D = ROOT / "data_part2" / "emerging"
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 200)
AGE = {"motor": "driver_age_band", "buildings": "policyholder_age_band", "contents": "policyholder_age_band"}

def h(t):
    print(f"\n{'=' * 8} {t}")

def shares(df, col, by):
    return pd.crosstab(by, df[col].astype(str), normalize="index").round(3)

cust = pd.read_parquet(D / "customers.parquet")
addr = pd.read_parquet(D / "addresses.parquet")
claims = load_claims(D)

for line in LINES:
    p, t = load_policies(line, "train", D), load_policies(line, "test", D)
    t = t.assign(year=2025)
    a = pd.concat([p, t], ignore_index=True)
    a = a.merge(cust[["customer_id", "customer_since_year"]], on="customer_id", how="left") \
         .merge(addr[["address_id", "region"]], on="address_id", how="left")
    m = match_claims(claims[claims.line == line], p, line, True, cap=None)
    p = p.assign(n_claims=p.row_id.map(m.groupby("row_id").size()).fillna(0))

    h(f"{line.upper()}  rows and exposure by year")
    print(a.groupby("year").agg(rows=("row_id", "size"), exposure=("exposure_clean", "sum"),
                                policies=("policy_id", "nunique")).round(0))

    h(f"{line}: 1 portfolio shift (shares by year; 2025 = test)")
    for col in ["new_business", "channel", AGE[line], "region", "payment_method"]:
        if col in a:
            print(f"-- {col}\n{shares(a, col, a.year)}")
    print("-- customer_since_year\n", shares(a.assign(cs=a.customer_since_year.clip(lower=2015)), "cs", a.year))
    print("-- policy seen in training data (test rows)\n",
          a[a.year == 2025].policy_id.isin(p.policy_id).mean().round(3))

    h(f"{line}: 2 renewal_count")
    print(a.groupby("year").renewal_count.describe().round(2))
    print("-- train: renewal_count (capped 10) by new_business\n",
          pd.crosstab(p.new_business, p.renewal_count.clip(upper=10)))
    print("-- test: renewal_count (capped 10) by new_business\n",
          pd.crosstab(t.new_business, t.renewal_count.clip(upper=10)))
    r = p[p.year == 2023].groupby("policy_id").renewal_count.max().rename("rc23").to_frame() \
        .join(p[p.year == 2024].groupby("policy_id").renewal_count.max().rename("rc24"), how="inner")
    print("-- same policy, 2024 minus 2023 renewal_count\n", (r.rc24 - r.rc23).value_counts().head(6))

    h(f"{line}: 3 missing values")
    miss_cols = [c for c in p.columns if p[c].isna().mean() > 0.01 and c != "row_id"]
    if miss_cols:
        nm = p[miss_cols].isna()
        print("-- share missing\n", nm.mean().round(3).to_string())
        print("-- number of these columns missing per row\n",
              nm.sum(axis=1).value_counts(normalize=True).round(3).to_string())
        print("-- correlation of missing indicators\n", nm.astype(int).corr().round(2))
        any_m = nm.any(axis=1)
        print("-- missing share by year / channel / new_business\n",
              p.assign(m=any_m).groupby(["year", "channel", "new_business"]).m.mean().round(3).unstack())
        for c in miss_cols:
            g = p.assign(miss=p[c].isna()).groupby("miss").agg(exp=("exposure_clean", "sum"), n=("n_claims", "sum"))
            print(f"-- {c}: claims per exposure, present vs missing: "
                  f"{(g.n / g.exp).round(4).to_dict()}  (claims {g.n.astype(int).to_dict()})")

    h(f"{line}: 4 policy steps by start month (median)")
    q = a.assign(month=a.period_start.dt.to_period("M"), prem_py=a.written_premium / a.exposure_clean)
    cols = {"prem_py": "median"}
    if "sum_insured" in q:
        cols["sum_insured"] = "median"
    out = q.groupby("month").agg(rows=("row_id", "size"), **{k: (k, v) for k, v in cols.items()})
    if "sum_insured" in q:
        out["si_p99"] = q.groupby("month").sum_insured.quantile(0.99)
        out["si_max"] = q.groupby("month").sum_insured.max()
    print(out.round(0).to_string())


======== MOTOR  rows and exposure by year
       rows  exposure  policies
year                           
2023  21680   18865.0     20084
2024  25603   20104.0     23722
2025  90521   79007.0     83907

======== motor: 1 portfolio shift (shares by year; 2025 = test)
-- new_business
new_business  False   True
year                      
2023          0.879  0.121
2024          0.847  0.153
2025          0.240  0.760
-- channel
channel  broker  comparison_site  direct
year                                    
2023      0.701            0.049   0.249
2024      0.701            0.049   0.250
2025      0.700            0.050   0.250
-- driver_age_band
driver_age_band  17-24  25-34  35-49  50-64    65+
year                                              
2023             0.209  0.174  0.165  0.171  0.281
2024             0.191  0.206  0.158  0.164  0.282
2025             0.248  0.253  0.130  0.132  0.236
-- region
region    E01    E02    E03    E04    E05    E06
year                            

In [3]:
# Day 1 follow-up 2: new business vs renewals, sum_insured unit error, written_premium basis
import sys
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.home() / "IAA"
sys.path.insert(0, str(ROOT))
from src.target import LINES, load_claims, load_policies, match_claims
D, P1 = ROOT / "data_part2" / "emerging", ROOT / "data"
claims = load_claims(D)

def h(t):
    print(f"\n{'=' * 8} {t}")

for line in LINES:
    p, t = load_policies(line, "train", D), load_policies(line, "test", D)
    m = match_claims(claims[claims.line == line], p, line, True, cap=None)
    p = p.assign(n=p.row_id.map(m.groupby("row_id").size()).fillna(0),
                 net=p.row_id.map(m.groupby("row_id").net.sum()).fillna(0))

    h(f"{line}: frequency and net cost per exposure, by year and new_business")
    g = p.groupby(["year", "new_business"]).agg(exp=("exposure_clean", "sum"), claims=("n", "sum"), net=("net", "sum"))
    g["freq"], g["cost"] = g.claims / g.exp, g.net / g.exp
    print(g.round(3))
    h(f"{line}: by renewal_count band (train)")
    b = pd.cut(p.renewal_count, [-1, 0, 1, 3, 6, 10, 99], labels=["0", "1", "2-3", "4-6", "7-10", "11+"])
    g = p.groupby(b, observed=True).agg(exp=("exposure_clean", "sum"), claims=("n", "sum"), net=("net", "sum"))
    g["freq"], g["cost"] = g.claims / g.exp, g.net / g.exp
    print(g.round(3))

    h(f"{line}: written_premium vs exposure by start month (medians)")
    for name, df in [("emerging train", p)] + ([("PART 1 train", load_policies(line, "train", P1))] if P1.exists() else []):
        q = df.groupby(df.period_start.dt.month).agg(exposure=("exposure_clean", "median"),
                                                    written_premium=("written_premium", "median"))
        print(f"-- {name}\n", q.T.round(2).to_string())

    if "sum_insured" in p:
        h(f"{line}: sum_insured > 100x median")
        med = p.sum_insured.median()
        for name, df in [("train", p), ("test", t)]:
            big = df.sum_insured > 100 * med
            print(f"-- {name}: {big.sum()} rows ({big.mean():.3%}); divided by 1000:",
                  (df.sum_insured[big] / 1000).describe().round(0).to_dict())
        mb = m.merge(p[["row_id", "sum_insured"]], on="row_id")
        big = mb.sum_insured > 100 * med
        print("-- claims on those rows: incurred vs sum_insured / 1000\n",
              mb.loc[big, ["peril", "incurred_to_date"]].assign(si_div1000=mb.loc[big, "sum_insured"] / 1000).to_string())


======== motor: frequency and net cost per exposure, by year and new_business
                         exp  claims         net   freq     cost
year new_business                                               
2023 False         17655.000  3242.0  5868289.43  0.184  332.387
     True           1209.513   315.0   965650.63  0.260  798.380
2024 False         18273.642  3109.0  5767619.97  0.170  315.625
     True           1830.386   382.0   655028.24  0.209  357.863

======== motor: by renewal_count band (train)
                     exp  claims         net   freq     cost
renewal_count                                               
0               3039.899   697.0  1620678.87  0.229  533.136
1               3428.773   626.0  1215022.36  0.183  354.361
2-3             4399.781   766.0  1414130.02  0.174  321.409
4-6             6754.079  1145.0  1967534.35  0.170  291.311
7-10            9150.152  1697.0  3101147.50  0.185  338.918
11+            12195.858  2117.0  3938075.17  0.174  322.

In [4]:
# Is a very large sum_insured an entry error (x1000) or a real value?
import sys
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.home() / "IAA"
sys.path.insert(0, str(ROOT))
from src.target import load_policies
D = ROOT / "data_part2" / "emerging"
addr = pd.read_parquet(D / "addresses.parquet")

for line in ["buildings", "contents"]:
    p = pd.concat([load_policies(line, "train", D), load_policies(line, "test", D).assign(year=2025)])
    med = p.sum_insured.median()
    big = p.sum_insured > 100 * med
    print(f"\n======== {line}: {big.sum()} rows above 100x median ({med:,.0f})")

    # 3 gap in the distribution: how many rows in each size range (multiples of the median)
    r = pd.cut(p.sum_insured / med, [0, 2, 5, 10, 20, 50, 100, 500, 1000, 1e6])
    print("-- rows by sum_insured / median\n", r.value_counts().sort_index().to_string())

    # 4 shape of the number: divisible by 1000 after dividing by 1000?
    print("-- large values divisible by 1,000,000:", (p.loc[big, "sum_insured"] % 1_000_000 == 0).mean().round(3),
          "| normal values divisible by 1,000:", (p.loc[~big, "sum_insured"] % 1000 == 0).mean().round(3))

    # 2 premium on large rows vs others (annual written premium, not divided by exposure)
    print("-- written_premium, large vs normal rows\n",
          p.groupby(big.rename("large")).written_premium.describe().round(0).to_string())

    # 1 same policy in other years
    ids = p.loc[big, "policy_id"].unique()
    s = p[p.policy_id.isin(ids)].pivot_table(index="policy_id", columns="year", values="sum_insured", aggfunc="max")
    print(f"-- same policy across years ({len(ids)} policies; first 15)\n", s.head(15).to_string())
    both = s.notna().sum(axis=1) >= 2
    ratio = s[both].max(axis=1) / s[both].min(axis=1)
    print("-- policies seen in 2+ years: max/min ratio\n", ratio.describe().round(1).to_string())

    # 5 property characteristics
    cols = [c for c in ["property_type", "bedrooms"] if c in addr]
    a = p[["address_id"]].assign(large=big.values).merge(addr[["address_id"] + cols], on="address_id")
    for c in cols:
        print(f"-- {c}, large vs normal\n", pd.crosstab(a.large, a[c], normalize="index").round(2).to_string())


======== buildings: 157 rows above 100x median (236,000)
-- rows by sum_insured / median
 sum_insured
(0.0, 2.0]             77659
(2.0, 5.0]              1926
(5.0, 10.0]                0
(10.0, 20.0]               0
(20.0, 50.0]               0
(50.0, 100.0]              0
(100.0, 500.0]            12
(500.0, 1000.0]           69
(1000.0, 1000000.0]       76
-- large values divisible by 1,000,000: 1.0 | normal values divisible by 1,000: 1.0
-- written_premium, large vs normal rows
          count   mean    std    min    25%    50%    75%     max
large                                                           
False  79585.0  377.0  409.0   63.0  199.0  250.0  337.0  4717.0
True     157.0  376.0  400.0  102.0  198.0  250.0  342.0  2930.0
-- same policy across years (157 policies; first 15)
 year              2023         2024         2025
policy_id                                       
B00172584          NaN          NaN  364000000.0
B00177444          NaN          NaN  409000000.0


In [5]:
# Who are the 2025 new-business customers? Train NB vs test NB vs train renewals; NB effect within age band
import sys
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.home() / "IAA"
sys.path.insert(0, str(ROOT))
from src.target import LINES, load_claims, load_policies, match_claims
D, P1 = ROOT / "data_part2" / "emerging", ROOT / "data"
pd.set_option("display.width", 220, "display.max_columns", 40)
cust = pd.read_parquet(D / "customers.parquet")
claims = load_claims(D)
AGE = {"motor": "driver_age_band", "buildings": "policyholder_age_band", "contents": "policyholder_age_band"}

def h(t):
    print(f"\n{'=' * 8} {t}")

for line in LINES:
    p, t = load_policies(line, "train", D), load_policies(line, "test", D).assign(year=2025)
    a = pd.concat([p, t], ignore_index=True).merge(cust, on="customer_id", how="left")
    a["group"] = np.select([a.year == 2025, a.new_business], ["test NB" if True else "", "train NB"], "train renewal")
    a.loc[(a.year == 2025) & ~a.new_business, "group"] = "test renewal"
    a["start_month"] = a.period_start.dt.month

    h(f"{line}: 1 profile by group (means / shares)")
    num = [c for c in ["days_quote_to_inception", "missed_payments_12m", "discount_applied", "prior_claims_5y",
                       "licence_points", "years_at_address", "household_size", "ncb_years", "years_licensed",
                       "vehicle_value_group", "sum_insured", "exposure_clean"] if c in a]
    print(a.groupby("group")[num].mean().round(2).T.to_string())
    print("-- start in January\n", a.groupby("group").start_month.apply(lambda s: (s == 1).mean()).round(3).to_string())
    for c in [AGE[line], "channel", "previous_insurer", "campaign_code", "marketing_segment", "gender",
              "cover_type", "payment_method"]:
        if c in a:
            print(f"-- {c}\n", pd.crosstab(a.group, a[c].fillna("NA").astype(str), normalize="index").round(2).to_string())

    h(f"{line}: 2 NB effect within age band (train, claim frequency)")
    m = match_claims(claims[claims.line == line], p, line, True, cap=None)
    p = p.assign(n=p.row_id.map(m.groupby("row_id").size()).fillna(0))
    g = p.groupby([AGE[line], "new_business"]).agg(exp=("exposure_clean", "sum"), n=("n", "sum"))
    g["freq"] = g.n / g.exp
    f = g.freq.unstack()
    f["NB / renewal"] = f[True] / f[False]
    print(f.round(3).to_string())
    e = g.exp.unstack()
    w = e.sum(axis=1)
    print("-- age-standardised NB / renewal:", round((f[True] * w).sum() / (f[False] * w).sum(), 3))

    if P1.exists():
        h(f"{line}: 3 developed market, NB vs renewal frequency by year")
        p1 = load_policies(line, "train", P1)
        c1 = load_claims(P1)
        m1 = match_claims(c1[c1.line == line], p1, line, True, cap=None)
        p1 = p1.assign(n=p1.row_id.map(m1.groupby("row_id").size()).fillna(0))
        g1 = p1.groupby(["year", "new_business"]).agg(exp=("exposure_clean", "sum"), n=("n", "sum"))
        g1["freq"] = g1.n / g1.exp
        f1 = g1.freq.unstack()
        f1["NB / renewal"] = f1[True] / f1[False]
        print(f1.round(3).to_string())


======== motor: 1 profile by group (means / shares)
group                    test NB  test renewal  train NB  train renewal
days_quote_to_inception    13.16         13.15     13.31          13.11
missed_payments_12m         0.10          0.10      0.11           0.10
discount_applied            0.02          0.01      0.02           0.01
prior_claims_5y             0.26          0.27      0.27           0.27
licence_points              0.90          0.91      0.88           0.91
years_at_address            6.77          7.07      6.74           7.17
household_size              2.22          2.25      2.20           2.27
ncb_years                   5.69          7.17      5.45           6.51
years_licensed             21.52         27.57     20.84          27.84
vehicle_value_group        10.69         10.83     10.59          10.92
exposure_clean              0.88          0.84      0.47           0.88
-- start in January
 group
test NB          0.883
test renewal     0.935
train NB  

In [6]:
import pandas as pd
from pathlib import Path
D = Path.home() / "IAA" / "data_part2" / "emerging"
pd.set_option("display.width", 200, "display.max_columns", 60)
print(pd.read_parquet(D / "expenses.parquet").to_string())
r = pd.read_parquet(D / "reference_policies.parquet")
print(r.shape)
print(r[["row_id", "line", "channel", "exposure", "new_business"]].to_string())

                         item  developed  emerging                            unit
0    fixed_expense_per_policy      45.00     85.00                              CU
1            variable_expense       0.12      0.15                share of premium
2           commission_direct       0.00      0.00                share of premium
3  commission_comparison_site       0.10       NaN                share of premium
4           commission_broker       0.15      0.20                share of premium
5                 premium_tax       0.12      0.08                share of premium
6             cat_reinsurance       0.06      0.12  share of property risk premium
7        target_profit_margin       0.05      0.08                share of premium
(20, 50)
       row_id       line channel  exposure  new_business
0   M00161473      motor  broker    0.9644          True
1   M00176777      motor  broker    0.9918          True
2   M00185311      motor  direct    0.9644          True
3   M00167496   